# NBA Scout Data Processing

Notebook này dùng để thử nghiệm data processing trước khi chuyển logic vào codebase local.

Mục tiêu tạo 3 gold datasets:

- `player_role_features.parquet`: role profile và similarity features.
- `performance_training.parquet`: rolling form features và future production targets.
- `salary_training.parquet`: player-season salary analysis/training table.

Notebook cố tình không import code local trong `src/` hoặc `app/` để giữ giai đoạn exploration tách biệt.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
# Colab/bootstrap install. PyPI package name is nba-api; import path is nba_api.
%pip install -q nba-api

## 1. Paths and Input Contracts

Notebook ưu tiên đọc data từ Google Drive folder `My Drive/nba-scout-assistant/data`. Nếu chạy trên Colab, mount Drive trước bằng:

```python
from google.colab import drive
drive.mount("/content/drive")
```

Nếu chạy local và có Drive sync ở path khác, set env var `NBA_SCOUT_DATA_DIR` trỏ tới folder `data`.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

def first_existing_path(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None

env_data_dir = os.getenv("NBA_SCOUT_DATA_DIR")
drive_data_candidates = [
    Path("/content/drive/MyDrive/nba-scout-assistant/data"),
    Path("/content/drive/My Drive/nba-scout-assistant/data"),
    Path.home() / "Google Drive" / "My Drive" / "nba-scout-assistant" / "data",
    Path.home() / "Library" / "CloudStorage" / "GoogleDrive-MyDrive" / "nba-scout-assistant" / "data",
]

DATA_DIR = Path(env_data_dir).expanduser().resolve() if env_data_dir else first_existing_path(drive_data_candidates)
if DATA_DIR is None:
    DATA_DIR = PROJECT_ROOT / "data"
    print(f"Drive data folder not found. Falling back to local data folder: {DATA_DIR}")
else:
    print(f"Using data folder: {DATA_DIR}")

BRONZE_DIR = DATA_DIR / "bronze"
RAW_DIR = DATA_DIR / "raw"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
RAW_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)
GOLD_DIR.mkdir(parents=True, exist_ok=True)

PLAYERS_PATH = RAW_DIR / "players.parquet"
GAME_LOGS_PATH = RAW_DIR / "player_game_logs.parquet"
SEASON_STATS_PATH = RAW_DIR / "player_season_stats.parquet"
SALARY_CAP_PATH = RAW_DIR / "salary_cap_by_season.parquet"
PLAYER_SEASON_SALARIES_PATH = SILVER_DIR / "player_season_salaries.parquet"

OUTPUT_ROLE_FEATURES = GOLD_DIR / "player_role_features.parquet"
OUTPUT_PERFORMANCE_TRAINING = GOLD_DIR / "performance_training.parquet"
OUTPUT_SALARY_TRAINING = GOLD_DIR / "salary_training.parquet"

RAW_DIR, GOLD_DIR

In [ ]:
RAW_SCHEMAS = {
    "players": [
        "player_id", "player_name", "birth_date", "position", "height", "weight",
    ],
    "player_game_logs": [
        "player_id", "game_date", "game_id", "season", "team_id", "minutes", "points", "assists",
        "rebounds", "usage_pct", "true_shooting_pct", "opponent", "home_away", "rest_days",
    ],
    "player_season_stats": [
        "player_id", "season", "team_id", "age", "minutes", "usage_pct", "points_per_100",
        "assists_per_100", "rebounds_per_100", "true_shooting_pct", "three_point_attempt_rate",
        "free_throw_rate", "turnover_rate", "steal_rate", "block_rate", "offensive_rating",
        "defensive_rating",
    ],
    "salary_cap_by_season": ["season", "salary_cap"],
    "player_season_salaries": [
        "player_name", "team", "season_start_year", "season_end_year", "season_label", "salary_usd",
        "source", "source_file", "collected_at",
    ],
}

def read_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")

def report_schema(name: str, df: pd.DataFrame, expected_columns: list[str]) -> None:
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    missing = sorted(set(expected_columns) - set(df.columns))
    extra = sorted(set(df.columns) - set(expected_columns))
    if missing:
        print("  Missing expected columns:", missing)
    if extra:
        print("  Extra columns:", extra[:25], "..." if len(extra) > 25 else "")

## 2. Load Raw Tables

## 2A. Optional: Fetch Raw Data From `nba_api`

`nba_api` có thể lấy được player list, league-wide game logs, và league-wide season stats từ NBA Stats. Nếu `data/raw/` đang trống, notebook sẽ tự fetch các bảng này khi `AUTO_FETCH_NBA_API_IF_MISSING = True`.

Nguồn này không phải nguồn tốt cho salary history hoặc salary cap table. Salary history hiện dùng bảng Kaggle đã chuẩn hóa ở `data/silver/player_season_salaries.parquet`; nếu muốn phân tích theo cap share thì vẫn cần thêm `data/raw/salary_cap_by_season.parquet`.

In [ ]:
# Nếu kernel chưa có nba_api, chạy cell install `nba-api` ở đầu notebook rồi restart kernel nếu cần.

try:
    from nba_api.stats.endpoints import leaguedashplayerstats, playergamelogs
    from nba_api.stats.static import players as nba_static_players
    NBA_API_AVAILABLE = True
except ImportError:
    NBA_API_AVAILABLE = False
    print("nba_api is not installed. Run `%pip install nba-api` in this notebook if you want to fetch data.")

In [ ]:
AUTO_FETCH_NBA_API_IF_MISSING = True
OVERWRITE_EXISTING_NBA_API_RAW = False

# NBA season format used by stats.nba.com. Keep this aligned with salary data availability.
FETCH_SEASONS = [
    "2016-17", "2017-18", "2018-19", "2019-20", "2020-21",
    "2021-22", "2022-23", "2023-24", "2024-25",
]

# Temporal split for model experiments. Do not random-split these datasets.
TRAIN_END_SEASON = "2021-22"
VALIDATION_SEASONS = ["2022-23"]
TEST_SEASONS = ["2023-24"]
FINAL_HOLDOUT_SEASONS = ["2024-25"]
SEASON_TYPE = "Regular Season"
NBA_API_TIMEOUT_SECONDS = 180
NBA_API_MAX_RETRIES = 3

# NBA Stats can throttle requests. Increase this if calls fail intermittently.
REQUEST_SLEEP_SECONDS = 3.0
NBA_API_CACHE_DIR = RAW_DIR / "nba_api_cache"
NBA_API_CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def season_start_year(season: str) -> int:
    return int(str(season).split("-")[0])

def cache_safe_season(season: str) -> str:
    return str(season).replace("-", "_")

def nba_api_get_data_frame(endpoint_factory, label: str) -> pd.DataFrame:
    import time
    from requests.exceptions import ReadTimeout, Timeout

    last_error: Exception | None = None
    for attempt in range(1, NBA_API_MAX_RETRIES + 1):
        try:
            endpoint = endpoint_factory()
            return endpoint.get_data_frames()[0]
        except (ReadTimeout, Timeout, TimeoutError) as exc:
            last_error = exc
            wait_seconds = REQUEST_SLEEP_SECONDS * attempt
            print(f"Timeout while fetching {label}; retry {attempt}/{NBA_API_MAX_RETRIES} after {wait_seconds:.0f}s")
            time.sleep(wait_seconds)

    print(f"Failed to fetch {label}: {last_error}")
    return pd.DataFrame()

def normalize_nba_players() -> pd.DataFrame:
    if not NBA_API_AVAILABLE:
        return pd.DataFrame()

    raw_players = pd.DataFrame(nba_static_players.get_players())
    players_df = raw_players.rename(
        columns={
            "id": "player_id",
            "full_name": "player_name",
        }
    )
    players_df["birth_date"] = pd.NaT
    players_df["position"] = pd.NA
    players_df["height"] = pd.NA
    players_df["weight"] = pd.NA
    return players_df[["player_id", "player_name", "birth_date", "position", "height", "weight"]]

def normalize_player_game_logs(logs: pd.DataFrame) -> pd.DataFrame:
    if logs.empty:
        return pd.DataFrame()

    normalized = pd.DataFrame({
        "player_id": logs["PLAYER_ID"],
        "player_name": logs.get("PLAYER_NAME"),
        "game_date": pd.to_datetime(logs["GAME_DATE"]),
        "game_id": logs["GAME_ID"],
        "season": logs["REQUESTED_SEASON"],
        "team_id": logs["TEAM_ID"],
        "team_abbreviation": logs.get("TEAM_ABBREVIATION"),
        "minutes": logs["MIN"],
        "points": logs["PTS"],
        "assists": logs["AST"],
        "rebounds": logs["REB"],
        "usage_pct": pd.NA,
        "true_shooting_pct": pd.NA,
        "opponent": logs["MATCHUP"].astype(str).str.extract(r"(?:vs\.|@)\s+([A-Z]{2,3})", expand=False),
        "home_away": np.where(logs["MATCHUP"].astype(str).str.contains(" @ ", regex=False), "AWAY", "HOME"),
    })
    normalized = normalized.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    normalized["rest_days"] = normalized.groupby("player_id")["game_date"].diff().dt.days
    return normalized

def fetch_league_player_game_logs(seasons: list[str]) -> pd.DataFrame:
    if not NBA_API_AVAILABLE:
        return pd.DataFrame()

    frames = []
    for season in seasons:
        cache_path = NBA_API_CACHE_DIR / f"player_game_logs_{cache_safe_season(season)}.parquet"
        if cache_path.exists() and not OVERWRITE_EXISTING_NBA_API_RAW:
            print(f"Using cached player game logs: {season}")
            frames.append(pd.read_parquet(cache_path))
            continue

        print(f"Fetching player game logs: {season}")
        raw = nba_api_get_data_frame(
            lambda: playergamelogs.PlayerGameLogs(
                season_nullable=season,
                season_type_nullable=SEASON_TYPE,
                timeout=NBA_API_TIMEOUT_SECONDS,
            ),
            label=f"player game logs {season}",
        )
        if raw.empty:
            continue
        raw["REQUESTED_SEASON"] = season
        raw.to_parquet(cache_path, index=False)
        frames.append(raw)

    if not frames:
        return pd.DataFrame()

    return normalize_player_game_logs(pd.concat(frames, ignore_index=True))

def normalize_player_season_stats(stats: pd.DataFrame) -> pd.DataFrame:
    if stats.empty:
        return pd.DataFrame()

    def col(name: str, default: float | None = np.nan) -> pd.Series:
        if name in stats.columns:
            return stats[name]
        return pd.Series(default, index=stats.index)

    fga = col("FGA").replace(0, np.nan)
    normalized = pd.DataFrame({
        "player_id": stats["PLAYER_ID"],
        "player_name": col("PLAYER_NAME", pd.NA),
        "season": stats["REQUESTED_SEASON"],
        "team_id": stats["TEAM_ID"],
        "age": col("AGE"),
        "minutes": col("MIN"),
        "usage_pct": col("USG_PCT"),
        "points_per_100": col("PTS"),
        "assists_per_100": col("AST"),
        "rebounds_per_100": col("REB"),
        "true_shooting_pct": col("TS_PCT"),
        "three_point_attempt_rate": col("FG3A") / fga,
        "free_throw_rate": col("FTA") / fga,
        "turnover_rate": col("TM_TOV_PCT").fillna(col("TOV_PCT")),
        "steal_rate": col("STL_PCT"),
        "block_rate": col("BLK_PCT"),
        "offensive_rating": col("OFF_RATING"),
        "defensive_rating": col("DEF_RATING"),
    })
    return normalized.sort_values(["season", "player_id"]).reset_index(drop=True)

def fetch_league_player_season_stats(seasons: list[str]) -> pd.DataFrame:
    if not NBA_API_AVAILABLE:
        return pd.DataFrame()

    frames = []
    for season in seasons:
        cache_path = NBA_API_CACHE_DIR / f"player_season_stats_{cache_safe_season(season)}.parquet"
        if cache_path.exists() and not OVERWRITE_EXISTING_NBA_API_RAW:
            print(f"Using cached player season stats: {season}")
            frames.append(pd.read_parquet(cache_path))
            continue

        print(f"Fetching player season stats: {season}")
        base = nba_api_get_data_frame(
            lambda: leaguedashplayerstats.LeagueDashPlayerStats(
                season=season,
                season_type_all_star=SEASON_TYPE,
                per_mode_detailed="Per100Possessions",
                measure_type_detailed_defense="Base",
                timeout=NBA_API_TIMEOUT_SECONDS,
            ),
            label=f"player season base stats {season}",
        )
        advanced = nba_api_get_data_frame(
            lambda: leaguedashplayerstats.LeagueDashPlayerStats(
                season=season,
                season_type_all_star=SEASON_TYPE,
                per_mode_detailed="Per100Possessions",
                measure_type_detailed_defense="Advanced",
                timeout=NBA_API_TIMEOUT_SECONDS,
            ),
            label=f"player season advanced stats {season}",
        )
        if base.empty or advanced.empty:
            continue
        merge_keys = [key for key in ["PLAYER_ID", "PLAYER_NAME", "TEAM_ID", "TEAM_ABBREVIATION", "AGE"] if key in base.columns and key in advanced.columns]
        raw = base.merge(advanced, on=merge_keys, how="left", suffixes=("", "_ADV"))
        raw["REQUESTED_SEASON"] = season
        raw.to_parquet(cache_path, index=False)
        frames.append(raw)

    if not frames:
        return pd.DataFrame()

    return normalize_player_season_stats(pd.concat(frames, ignore_index=True))

In [ ]:
def should_fetch(path: Path) -> bool:
    return OVERWRITE_EXISTING_NBA_API_RAW or not path.exists()

if AUTO_FETCH_NBA_API_IF_MISSING:
    if not NBA_API_AVAILABLE:
        raise ImportError("Install nba_api first by running `%pip install nba-api`, then restart/re-run the notebook.")

    if should_fetch(PLAYERS_PATH):
        fetched_players = normalize_nba_players()
        fetched_players.to_parquet(PLAYERS_PATH, index=False)
        print(f"Saved {PLAYERS_PATH}: {fetched_players.shape}")
    else:
        print(f"Using existing {PLAYERS_PATH}")

    if should_fetch(GAME_LOGS_PATH):
        fetched_game_logs = fetch_league_player_game_logs(FETCH_SEASONS)
        fetched_game_logs.to_parquet(GAME_LOGS_PATH, index=False)
        print(f"Saved {GAME_LOGS_PATH}: {fetched_game_logs.shape}")
    else:
        print(f"Using existing {GAME_LOGS_PATH}")

    if should_fetch(SEASON_STATS_PATH):
        fetched_season_stats = fetch_league_player_season_stats(FETCH_SEASONS)
        fetched_season_stats.to_parquet(SEASON_STATS_PATH, index=False)
        print(f"Saved {SEASON_STATS_PATH}: {fetched_season_stats.shape}")
    else:
        print(f"Using existing {SEASON_STATS_PATH}")

missing_salary_sources = [path for path in [PLAYER_SEASON_SALARIES_PATH, SALARY_CAP_PATH] if not path.exists()]
if missing_salary_sources:
    print("Additional non-nba_api data status for salary analysis:")
    for path in missing_salary_sources:
        print(f"  {path}")

In [ ]:
players = read_table(PLAYERS_PATH)
game_logs = read_table(GAME_LOGS_PATH)
season_stats = read_table(SEASON_STATS_PATH)
salary_cap = read_table(SALARY_CAP_PATH)
player_season_salaries = read_table(PLAYER_SEASON_SALARIES_PATH)

tables = {
    "players": players,
    "player_game_logs": game_logs,
    "player_season_stats": season_stats,
    "salary_cap_by_season": salary_cap,
    "player_season_salaries": player_season_salaries,
}

for table_name, table_df in tables.items():
    report_schema(table_name, table_df, RAW_SCHEMAS[table_name])

## 3. Gold Dataset 1: Player Role Features

Một dòng = một cầu thủ trong một mùa. Dataset này phục vụ player similarity, candidate retrieval, role explanation, và có thể tái sử dụng cho salary model.

In [ ]:
ROLE_BASE_FEATURES = [
    "minutes",
    "usage_pct",
    "points_per_100",
    "assists_per_100",
    "rebounds_per_100",
    "true_shooting_pct",
    "three_point_attempt_rate",
    "free_throw_rate",
    "turnover_rate",
    "steal_rate",
    "block_rate",
    "offensive_rating",
    "defensive_rating",
]

def add_role_dimensions(df: pd.DataFrame) -> pd.DataFrame:
    role = df.copy()
    role["scoring_creation"] = role[["points_per_100", "usage_pct", "free_throw_rate"]].mean(axis=1)
    role["playmaking"] = role[["assists_per_100", "usage_pct"]].mean(axis=1) - role["turnover_rate"].fillna(0)
    role["shooting"] = role[["true_shooting_pct", "three_point_attempt_rate"]].mean(axis=1)
    role["rim_pressure"] = role[["free_throw_rate", "points_per_100"]].mean(axis=1)
    role["rebounding"] = role["rebounds_per_100"]
    role["perimeter_defense"] = role["steal_rate"]
    role["interior_defense"] = role["block_rate"]
    role["two_way_impact"] = role["offensive_rating"] - role["defensive_rating"]
    return role

ROLE_DIMENSIONS = [
    "scoring_creation", "playmaking", "shooting", "rim_pressure", "rebounding",
    "perimeter_defense", "interior_defense", "two_way_impact",
]

def build_player_role_features(season_stats_df: pd.DataFrame, players_df: pd.DataFrame) -> pd.DataFrame:
    if season_stats_df.empty:
        return pd.DataFrame()

    required = ["player_id", "season", "team_id", "age"] + ROLE_BASE_FEATURES
    role = season_stats_df[required].copy()
    role = add_role_dimensions(role)

    identity_cols = [col for col in ["player_id", "player_name", "position"] if col in players_df.columns]
    if identity_cols:
        role = role.merge(players_df[identity_cols].drop_duplicates("player_id"), on="player_id", how="left")

    output_cols = [
        "player_id", "player_name", "season", "team_id", "age", "position",
        *ROLE_BASE_FEATURES,
        *ROLE_DIMENSIONS,
    ]
    output_cols = [col for col in output_cols if col in role.columns]
    return role[output_cols].sort_values(["season", "player_id"]).reset_index(drop=True)

player_role_features = build_player_role_features(season_stats, players)
player_role_features.head()

In [ ]:
if not player_role_features.empty:
    player_role_features.to_parquet(OUTPUT_ROLE_FEATURES, index=False)
    print(f"Saved {OUTPUT_ROLE_FEATURES} with shape {player_role_features.shape}")

### Quick Similarity Prototype

Cell này giúp kiểm tra nhanh câu hỏi: cầu thủ nào có style/role gần một cầu thủ target. Có thể đổi `TARGET_PLAYER_NAME`, `TARGET_SEASON`, và `SIMILARITY_WEIGHTS`.

In [ ]:
TARGET_PLAYER_NAME = "LeBron James"
TARGET_SEASON = None

SIMILARITY_WEIGHTS = {
    "scoring_creation": 1.0,
    "playmaking": 1.2,
    "shooting": 0.8,
    "rim_pressure": 1.0,
    "rebounding": 0.7,
    "perimeter_defense": 0.8,
    "interior_defense": 0.4,
    "two_way_impact": 0.8,
}

def find_similar_players(
    role_df: pd.DataFrame,
    target_player_name: str,
    target_season: str | int | None = None,
    top_k: int = 10,
    weights: dict[str, float] | None = None,
) -> pd.DataFrame:
    if role_df.empty:
        return pd.DataFrame()

    features = [feature for feature in ROLE_DIMENSIONS if feature in role_df.columns]
    matrix = role_df[features].copy()
    if weights:
        for feature, weight in weights.items():
            if feature in matrix.columns:
                matrix[feature] = matrix[feature] * weight

    preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    X = preprocessor.fit_transform(matrix)

    target_mask = role_df["player_name"].str.lower().eq(target_player_name.lower())
    if target_season is not None:
        target_mask &= role_df["season"].eq(target_season)
    if not target_mask.any():
        raise ValueError(f"Target player not found: {target_player_name}, season={target_season}")

    target_idx = role_df[target_mask].index[-1]
    sims = cosine_similarity(X[target_idx : target_idx + 1], X).ravel()

    results = role_df.copy()
    results["similarity_score"] = sims
    results = results.loc[results.index != target_idx]
    return results.sort_values("similarity_score", ascending=False).head(top_k)

if not player_role_features.empty and "player_name" in player_role_features.columns:
    similar_players = find_similar_players(
        player_role_features,
        TARGET_PLAYER_NAME,
        target_season=TARGET_SEASON,
        top_k=10,
        weights=SIMILARITY_WEIGHTS,
    )
    display(similar_players[["player_name", "season", "team_id", "position", "similarity_score", *ROLE_DIMENSIONS]])

## 4. Gold Dataset 2: Performance Training

Một dòng = một cầu thủ tại một `as_of_date`. Features chỉ dùng dữ liệu trước hoặc tại `as_of_date`; targets dùng trung bình 5 trận tiếp theo.

In [ ]:
def add_rolling_player_features(game_logs_df: pd.DataFrame) -> pd.DataFrame:
    if game_logs_df.empty:
        return pd.DataFrame()

    df = game_logs_df.copy()
    df["game_date"] = pd.to_datetime(df["game_date"])
    df = df.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    grouped = df.groupby("player_id", group_keys=False)

    stat_prefixes = {"points": "pts", "assists": "ast", "rebounds": "reb"}

    for stat, prefix in stat_prefixes.items():
        for window in [5, 10, 20]:
            df[f"{prefix}_last_{window}"] = grouped[stat].transform(
                lambda s: s.shift(1).rolling(window=window, min_periods=1).mean()
            )
        df[f"{prefix}_season_to_date"] = grouped[stat].transform(
            lambda s: s.shift(1).expanding(min_periods=1).mean()
        )

    for stat in ["minutes", "usage_pct", "true_shooting_pct"]:
        df[f"{stat}_last_5"] = grouped[stat].transform(
            lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
        )
        df[f"{stat}_last_10"] = grouped[stat].transform(
            lambda s: s.shift(1).rolling(window=10, min_periods=1).mean()
        )
        df[f"{stat}_trend"] = df[f"{stat}_last_5"] - df[f"{stat}_last_10"]

    for stat in ["points", "assists", "rebounds"]:
        df[f"target_next_5_games_{stat}"] = grouped[stat].transform(
            lambda s: s.shift(-1).rolling(window=5, min_periods=1).mean().shift(-4)
        )

    df = df.rename(columns={"game_date": "as_of_date"})
    return df

def build_performance_training(game_logs_df: pd.DataFrame) -> pd.DataFrame:
    features = add_rolling_player_features(game_logs_df)
    if features.empty:
        return pd.DataFrame()

    id_cols = [
        "player_id", "as_of_date", "game_id", "season", "team_id", "opponent", "home_away", "rest_days",
    ]
    feature_cols = [
        "pts_last_5", "pts_last_10", "pts_last_20", "pts_season_to_date",
        "ast_last_5", "ast_last_10", "ast_last_20", "ast_season_to_date",
        "reb_last_5", "reb_last_10", "reb_last_20", "reb_season_to_date",
        "minutes_last_5", "minutes_last_10", "minutes_trend",
        "usage_pct_last_5", "usage_pct_last_10", "usage_pct_trend",
        "true_shooting_pct_last_5", "true_shooting_pct_last_10", "true_shooting_pct_trend",
    ]
    target_cols = [
        "target_next_5_games_points", "target_next_5_games_assists", "target_next_5_games_rebounds",
    ]
    output_cols = [col for col in [*id_cols, *feature_cols, *target_cols] if col in features.columns]
    return features[output_cols].dropna(subset=target_cols, how="all").reset_index(drop=True)

performance_training = build_performance_training(game_logs)
performance_training.head()

In [ ]:
if not performance_training.empty:
    performance_training.to_parquet(OUTPUT_PERFORMANCE_TRAINING, index=False)
    print(f"Saved {OUTPUT_PERFORMANCE_TRAINING} with shape {performance_training.shape}")

## 5. Gold Dataset 3: Salary Analysis / Training

Một dòng = một cầu thủ trong một mùa lương. Dataset này bỏ qua contract signing/term và tập trung vào phân tích `salary_usd` theo mùa, có thể join với role features để học quan hệ giữa production/role và salary.

Nếu có `salary_cap_by_season.parquet`, notebook sẽ thêm `salary_cap_share = salary_usd / salary_cap`. Nếu chưa có, vẫn phân tích được salary USD.

In [ ]:
def normalize_name_for_join(value: object) -> str | None:
    if pd.isna(value):
        return None
    return str(value).strip().lower().replace(".", "").replace("'", "")

def build_salary_training(
    player_salaries_df: pd.DataFrame,
    salary_cap_df: pd.DataFrame,
    role_features_df: pd.DataFrame,
) -> pd.DataFrame:
    if player_salaries_df.empty:
        return pd.DataFrame()

    salary = player_salaries_df.copy()
    salary["player_name_join"] = salary["player_name"].map(normalize_name_for_join)
    salary["target_salary_usd"] = salary["salary_usd"]

    if not salary_cap_df.empty:
        cap = salary_cap_df.copy()
        if "season" in cap.columns and "season_label" not in cap.columns:
            cap = cap.rename(columns={"season": "season_label"})
        salary = salary.merge(cap[["season_label", "salary_cap"]], on="season_label", how="left")
        salary["salary_cap_share"] = salary["salary_usd"] / salary["salary_cap"]
    else:
        salary["salary_cap"] = pd.NA
        salary["salary_cap_share"] = pd.NA

    if not role_features_df.empty and "player_name" in role_features_df.columns:
        role = role_features_df.copy()
        role["player_name_join"] = role["player_name"].map(normalize_name_for_join)
        role = role.rename(columns={"season": "season_label"})
        keep_cols = [
            "player_name_join", "season_label", "player_id", "team_id", "age", "position",
            *ROLE_BASE_FEATURES, *ROLE_DIMENSIONS,
        ]
        keep_cols = [col for col in keep_cols if col in role.columns]
        salary = salary.merge(
            role[keep_cols].drop_duplicates(["player_name_join", "season_label"]),
            on=["player_name_join", "season_label"],
            how="left",
        )

    output_cols = [
        "player_id", "player_name", "team", "team_id", "season_start_year", "season_end_year",
        "season_label", "age", "position", "salary_usd", "salary_cap", "salary_cap_share",
        "target_salary_usd", "source", "source_file", "collected_at",
        *ROLE_BASE_FEATURES, *ROLE_DIMENSIONS,
    ]
    output_cols = [col for col in output_cols if col in salary.columns]
    return salary[output_cols].sort_values(["season_start_year", "salary_usd", "player_name"], ascending=[True, False, True]).reset_index(drop=True)

salary_training = build_salary_training(player_season_salaries, salary_cap, player_role_features)
salary_training.head()

In [ ]:
if not salary_training.empty:
    salary_training.to_parquet(OUTPUT_SALARY_TRAINING, index=False)
    print(f"Saved {OUTPUT_SALARY_TRAINING} with shape {salary_training.shape}")

## 6. Basic Data Quality Checks

Các check này chỉ để exploration. Khi logic ổn, có thể chuyển thành test hoặc Great Expectations suite sau.

In [ ]:
def quality_summary(name: str, df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        print(f"{name}: empty")
        return pd.DataFrame()
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean(),
        "n_unique": df.nunique(dropna=True),
    })
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    return summary.sort_values("missing_pct", ascending=False)

quality_summary("player_role_features", player_role_features).head(20)

In [ ]:
quality_summary("performance_training", performance_training).head(20)

In [ ]:
quality_summary("salary_training", salary_training).head(20)

## 7. Temporal Splits

Vì đây là bài toán theo thời gian, không random split. Các mùa gần nhất nên được giữ lại để validation/test/holdout, nhất là khi salary data chỉ đến `2024-25`.

In [ ]:
def assign_temporal_split(season_label: object) -> str:
    if pd.isna(season_label):
        return "unknown"
    season = str(season_label)
    if season in FINAL_HOLDOUT_SEASONS:
        return "final_holdout"
    if season in TEST_SEASONS:
        return "test"
    if season in VALIDATION_SEASONS:
        return "validation"
    if season <= TRAIN_END_SEASON:
        return "train"
    return "future_or_unassigned"

if not salary_training.empty and "season_label" in salary_training.columns:
    salary_training = salary_training.copy()
    salary_training["split"] = salary_training["season_label"].map(assign_temporal_split)
    salary_training.to_parquet(OUTPUT_SALARY_TRAINING, index=False)
    display(salary_training["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="rows"))

if not performance_training.empty and "season" in performance_training.columns:
    performance_training = performance_training.copy()
    performance_training["split"] = performance_training["season"].map(assign_temporal_split)
    performance_training.to_parquet(OUTPUT_PERFORMANCE_TRAINING, index=False)
    display(performance_training["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="rows"))

## 8. Next Decisions

- Chốt raw data source thực tế cho `players`, `player_game_logs`, `player_season_stats`, `player_season_salaries`, `salary_cap_by_season`.
- Chốt cách join salary với stats: cùng mùa để phân tích mô tả, hoặc mùa trước nếu dùng làm model dự đoán salary tương lai.
- Chốt role dimensions: công thức hiện tại chỉ là baseline heuristic để exploration, chưa phải product formula cuối.
- Chốt target horizon cho performance forecast: 5 games, 10 games, hoặc remainder-of-season.
- Giữ `2024-25` làm final holdout khi có đủ feature/salary overlap; không dùng để chọn model.
- Sau khi notebook chạy ổn, chuyển từng function thành module xử lý dữ liệu có test.